# Session 13 — 3/3: push the batch, scan depth, build the report

Run 3 takes the winning reversible variant and pushes the batch size to the
edge of the GPU. Then a depth scan isolates the actual claim of reversibility:
activation memory that does not grow with the number of layers.

In [1]:
!nvidia-smi
!git clone https://github.com/rjvim/era-v5-session13-reversibility repo 2>/dev/null || (cd repo && git pull)
%cd repo
!pip -q install -r requirements.txt
import sys; sys.path.insert(0, 'src')

Wed Sep 23 19:11:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   44C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os
if not os.path.exists('data/train.bin'):
    !python src/data.py --out_dir data --tokens 50000000

README.md: 100% 26.4k/26.4k [00:00<00:00, 58.8MB/s]
Resolving data files: 100% 2410/2410 [00:00<00:00, 24823.61it/s]
{'n_train': 50000000, 'n_val': 1000000, 'dataset': 'HuggingFaceFW/fineweb-edu', 'config': 'sample-10BT', 'encoding': 'gpt2'}


## Max batch for each mode

In [3]:
!python src/train.py --mode midpoint --find_max_batch --seq_len 512
!python src/train.py --mode euler    --find_max_batch --seq_len 512

  batch 1: ok peak 0.57GB
  batch 2: ok peak 0.85GB
  batch 4: ok peak 1.43GB
  batch 8: ok peak 2.59GB
  batch 16: ok peak 4.90GB
  batch 32: ok peak 9.53GB
  batch 64: ok peak 18.79GB
  batch 128: OOM
  batch 96: OOM
  batch 80: OOM
  batch 72: ok peak 21.10GB
  batch 76: OOM
  batch 74: ok peak 21.68GB
  batch 75: OOM
{
  "mode": "midpoint",
  "seq_len": 512,
  "max_batch": 74,
  "peak_mem_gb_at_max": 21.68362855911255,
  "device": "NVIDIA L4",
  "dtype": "bf16",
  "h": 0.5
}
  batch 1: ok peak 0.56GB
  batch 2: ok peak 0.85GB
  batch 4: ok peak 1.43GB
  batch 8: ok peak 2.59GB
  batch 16: ok peak 4.90GB
  batch 32: ok peak 9.52GB
  batch 64: ok peak 18.76GB
  batch 128: OOM
  batch 96: OOM
  batch 80: OOM
  batch 72: ok peak 21.07GB
  batch 76: OOM
  batch 74: OOM
  batch 73: ok peak 21.36GB
{
  "mode": "euler",
  "seq_len": 512,
  "max_batch": 73,
  "peak_mem_gb_at_max": 21.357740879058838,
  "device": "NVIDIA L4",
  "dtype": "bf16",
  "h": 0.5
}


## Run 3 — midpoint @ max batch

In [4]:
import json
BATCH_MAX = json.load(open('results/maxbatch_midpoint.json'))['max_batch']
BATCH_FIX = json.load(open('results/maxbatch_baseline.json'))['max_batch']
print(f'baseline max {BATCH_FIX} -> midpoint max {BATCH_MAX} ({BATCH_MAX/BATCH_FIX:.2f}x)')
!python src/train.py --mode midpoint --batch_size {BATCH_MAX} --run_name midpoint_max     --h 0.5 --check_recon --seq_len 512 --total_tokens 50000000 --resume


baseline max 50 -> midpoint max 74 (1.48x)
step      0/1319 loss 10.8850 lr 2.31e-05 33,828 tok/s peak 21.52GB
Traceback (most recent call last):
  File "/content/repo/src/train.py", line 330, in <module>
    run(args)
    ~~~^^^^^^
  File "/content/repo/src/train.py", line 146, in run
    (loss / args.grad_accum).backward()
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "/usr/local/lib/python3.13/dist-packages/torch/_tensor.py", line 631, in backward
    torch.autograd.backward(
    ~~~~~~~~~~~~~~~~~~~~~~~^
        self, gradient, retain_graph, create_graph, inputs=inputs
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/torch/autograd/__init__.py", line 381, in backward
    _engine_run_backward(
    ~~~~~~~~~~~~~~~~~~~~^
        tensors,
        ^^^^^^^^
    ...<5 lines>...
        accumulate_grad=True,
        ^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/torch/autograd/gr

## Memory vs depth

Same batch and sequence length, layers swept. Baseline should grow roughly linearly; midpoint should stay flat.

In [5]:
!python scripts/depth_scan.py --layers 4 8 16 32 64 --batch_size 8 --seq_len 512

{'n_layer': 4, 'baseline': 2.6763553619384766, 'midpoint': 2.536947250366211}
{'n_layer': 8, 'baseline': 2.8587779998779297, 'midpoint': 2.5721492767333984}
{'n_layer': 16, 'baseline': 3.223623275756836, 'midpoint': 2.6425533294677734}
{'n_layer': 32, 'baseline': 3.9533138275146484, 'midpoint': 2.7833614349365234}
{'n_layer': 64, 'baseline': 5.412694931030273, 'midpoint': 3.0649776458740234}
wrote depth_scaling.json


## Cost comparison

What the memory saving is worth in rupees/dollars: slower per token, but a smaller GPU tier or fewer nodes.

In [6]:
import json
b = json.load(open('results/baseline_fixed.json'))
m = json.load(open('results/midpoint_fixed.json'))
PRICE_PER_HR = 0.35   # <-- set to the actual price of the GPU tier you used
for name, d in [('baseline', b), ('midpoint', m)]:
    hrs = d['tokens_trained'] / d['tok_per_s'] / 3600
    print(f"{name:9s} {hrs:.2f} GPU-hours  ${hrs*PRICE_PER_HR:.2f} for "
          f"{d['tokens_trained']:,} tokens  (peak {d['peak_mem_gb']:.2f} GB)")
print('\nreversibility costs %.1f%% more compute-time for %.2fx the peak memory'
      % (100*(b['tok_per_s']/m['tok_per_s']-1), m['peak_mem_gb']/b['peak_mem_gb']))

baseline  0.17 GPU-hours  $0.06 for 49,996,800 tokens  (peak 16.94 GB)
midpoint  0.18 GPU-hours  $0.06 for 49,996,800 tokens  (peak 14.74 GB)

reversibility costs 11.8% more compute-time for 0.87x the peak memory


## Generate the README and verify it

In [8]:
!git add -A
!git -c user.email=rajivs.iitkgp@gmail.com -c user.name=rjvim commit -q -m 'session 13: results + generated README'
!git push


fatal: could not read Username for 'https://github.com': No such device or address
